# RSA — 02: Model Fits and Statistics

Computes the fit between neural RDMs and theoretical model RDMs over time,
then runs cluster-based permutation tests to identify significant windows.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from eeg_toolkit import load_config
from eeg_toolkit.rsa import load_all_rdms, compute_model_fits, square_to_vec
from eeg_toolkit.rsa_stats import test_model_fit, compare_model_fits, plot_model_fits

# ── Update this path ──
cfg = load_config('../../configs/your_experiment.yaml')

group = load_all_rdms(cfg, 'your_window', 'my_rsa')
times = group['times']
cond_names = sorted(group['condition_names'])

print("Setup OK")

In [ ]:
# ── Reconstruct model RDMs ──
# ── Copy from notebook 01 — must match exactly ──
n_cond = len(cond_names)
feature_values = np.array([1, 1, 2, 2])   # update to match your features

models = {
    'Feature identity': (feature_values[:, None] != feature_values[None, :]).astype(float),
    'Feature distance': np.abs(feature_values[:, None] - feature_values[None, :]).astype(float),
}

In [ ]:
# ── Compute model fits (Spearman r over time) ──
fits = compute_model_fits(group, models)

In [ ]:
# ── Run cluster-based permutation tests for each model ──
stats = {}
for name, fit_arr in fits.items():
    print(f"\n{'='*60}")
    stats[name] = test_model_fit(
        fit_arr, times,
        model_name=name,
        n_permutations=10000,
    )

In [ ]:
# ── Plot model fits with significance bars ──
# ── Update colors to match your model palette ──
colors = {
    'Feature identity': '#2166AC',
    'Feature distance': '#B2182B',
}

n_subj = next(iter(fits.values())).shape[0]
fig, ax = plt.subplots(figsize=(10, 5))

for name, fit_arr in fits.items():
    mean_r = np.nanmean(fit_arr, axis=0)
    sem_r  = np.nanstd(fit_arr, axis=0) / np.sqrt(n_subj)
    col = colors.get(name, None)
    ax.plot(times, mean_r, color=col, lw=2, label=name)
    ax.fill_between(times, mean_r - sem_r, mean_r + sem_r, color=col, alpha=0.15)

ax.axhline(0, color='k', lw=0.8, ls=':')
ax.axvline(0, color='k', lw=0.8, ls=':')

# Significance bars
y_min, y_max = ax.get_ylim()
bar_gap = (y_max - y_min) * 0.03
for mi, (name, results) in enumerate(stats.items()):
    col = colors.get(name, 'gray')
    y_pos = y_min - bar_gap * (mi + 1)
    for cluster in results.get('significant_clusters', []):
        t0 = times[cluster['start_idx']]
        t1 = times[cluster['end_idx']]
        ax.plot([t0, t1], [y_pos, y_pos], color=col, lw=3.5,
                solid_capstyle='butt', alpha=0.85)

ax.set_xlabel('Time (s)')
ax.set_ylabel('Spearman r')
ax.set_xlim(times[0], times[-1])
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()